# 07 — Robust QUBO construction (F1)

## Question

How is the scenario-averaged robust QUBO F1 constructed from the K=8 scenarios, and how is the corrected scenario transformation (Stage 6) applied?

## Why this test exists

The naive scenario transformation (Stage 5 prose) modified R_i but left the variable set fixed. This was wrong: the QUBO has no intrinsic incentive to use early slots, so a later-slot schedule could be preferred even when the scenario's window has shrunk. The corrected transformation (Stage 6 Part D) keeps the variable set fixed but applies a per-scenario diagonal penalty `M_window = 1e6` to every slot outside the scenario's effective window. This is a pure-QUBO soft-exclusion: the QUBO is forced to put x[i,t] = 0 for out-of-window slots without introducing auxiliary variables.

## Method

For each scenario s with centroid (Δd_s, ΔE_s):

1. Recompute the effective deadline: `d_eff = d_slot − floor(Δd_s / Δ)`.
2. Build a per-scenario QUBO `Q_s` on the BASE variable set (no    auxiliary variables).
3. Add M_window to the diagonal of every slot t > d_eff.
4. The robust QUBO is `Q_robust = Σ_s p_s · Q_s`, with p_s = 1/K.

Algebraic validation: enumerate all 2^11 bitstrings, verify that F_robust_QUBO(x) = Σ_s p_s · F_s_independent(x) + C, where F_s is the original objective on the base instance with the scenario's R_i and M_window contribution.

**FROZEN CONFIGURATION.** The methodology is frozen at `artifacts/final_experiment_config.json` (version `stage7.v1`). K=8, α=1.0, M_window=1e6, ρ_d=1.0, ρ_p=0.1, ρ_cap=0.5, P_target=6.6 kW, P_site_max=9.9 kW, Δ=15 min, calibration window 2018-05-01..2019-07-01, held-out window 2019-07-01..2020-01-01, QAOA p=1 / COBYLA / seeds [0,1,2] / shots 1024. No parameter may be modified based on held-out results.


## Implementation


In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from stage6.robust_qaoa import (validate_corrected_robust_qubo,
                                    build_f0_deterministic, build_f1_robust,
                                    build_f2_adopt, M_window_penalty)
from stage5.uncertainty import placeholder_uncertainty, kmeans_joint, compute_robust_rho_d
from stage3.ev_scheduling import toy_instance
import numpy as np

samples, _ = placeholder_uncertainty(seed=20260829)
cal = [s for s in samples if s.calibration]
dd = np.array([s.delta_d_minutes for s in cal if s.delta_d_minutes is not None])
de = np.array([s.delta_e_kwh for s in cal if s.delta_e_kwh is not None])
sc = kmeans_joint(dd, de, K=8, seed=20260829+8)['clusters']
inst = toy_instance('toy_B_3x4', N=3, T=4)

val = validate_corrected_robust_qubo(inst, sc, rho_d=1.0, rho_p=0.1, rho_cap=0.5, tol=1e-7)
print(f"M_window = {M_window_penalty:.0e}")
print(f"Corrected robust QUBO validation: max abs deviation = {val['max_abs_deviation_from_mean']:.2e}")
print(f"  passes: {val['passes']}")
print()
f0 = build_f0_deterministic(inst, 1.0, 0.1, 0.5)
f1 = build_f1_robust(inst, sc, 1.0, 0.1, 0.5)
rho_d_robust, adopt_stats = compute_robust_rho_d(1.0, inst, cal, alpha=1.0)
f2 = build_f2_adopt(inst, sc, 1.0, 0.1, 0.5, adopt_stats['gamma'])
print(f"F0 exact optimum: {f0.classical_optimum:.4f}")
print(f"F1 exact optimum: {f1.classical_optimum:.4f}")
print(f"F2 exact optimum: {f2.classical_optimum:.4f}  (γ = {adopt_stats['gamma']:.4f})")


## Result (placeholder)

The corrected robust QUBO passes the algebraic validation: the scenario-averaged QUBO and the sum of per-scenario original objectives differ by a constant offset, within 1e-7 tolerance. The F1 exact optimum is 0.0001 above the F0 optimum (negligible); the F2 optimum is 5.41 above (the ADOPT scaling makes F2 more deadline-conservative). The **placeholder** result is that F0 dominates F1/F2 in P(feasible); this is a placeholder-specific result driven by the dominant cluster's centroid, **not** a fundamental flaw (see Stage 7 §1).

## Interpretation

The robust QUBO algebra is exact. The schedule choice (which F0/F1/F2 wins) depends on the empirical cluster distribution, which on the placeholder favors F0. On real ACN-Data, the answer may differ; the methodology is set up to let the data speak.

## Limitations

- The placeholder F0-dominates result is real but placeholder-  specific. It does not predict the ACN-Data result.
- The M_window diagonal penalty is a soft constraint; for scenarios   with very tight windows it is not strictly binding. The Stage 6   Part D math shows that it forces the optimum to the in-window   subspace up to a constant offset.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.
